# ScreamingFace ↔ URL4 engine

Combine three model routes into one URL4-backed fusion, run it on deterministic benchmark
questions, and measure whether the majority beats the best individual model.

This uses deterministic model routes through the **real URL4 HTTP engine**. Before running the
notebook, start them from `packages/screamingface`:

```bash
./scripts/dev-url4.sh
```

To fetch GPQA Diamond instead of the bundled fixture, first accept its gated dataset terms and be
logged in to Hugging Face, then select the live dataset with `sf.config(mode="live")`. Your URL4
engine must also expose production-backed model routes.

The saved run uses deterministic routes, so its result is reproducible and makes no
provider-quality claim.

## 1 · Import the SDK

In [1]:
import screamingface as sf

# Optional: point the SDK at a hosted engine instead of the localhost default.
# sf.config("https://url4.example")

## 2 · Compose a fusion — Python or YAML

These are two representations of the same fusion. Use Python while exploring; use YAML when you
want a small configuration file to review or share.

### Option A · Python

In [2]:
fusion = sf.Fusion(
    "frontier-trio",
    models=[
        "codex/gpt-5.5",
        "gemini-cli/gemini-2.5-pro",
        "anthropic/claude-sonnet-4-6",
    ],
    reducer=sf.MajorityVote(tie_breaker="codex/gpt-5.5"),
)
fusion

Role,Model
Tie breaker,codex/gpt-5.5
Model,gemini-cli/gemini-2.5-pro
Model,anthropic/claude-sonnet-4-6


A plain string is shorthand for `{"model": "provider/model"}`. Mix in a strict
dictionary only when one model needs its own `name`, `prompt`, or URL4 `params`:

```python
models=[
    "openai/gpt-5.5",
    {
        "model": "anthropic/claude-opus-4.8",
        "name": "opus-sample-1",
        "params": {"temperature": 0.7},
    },
]
```

ScreamingFace validates these dictionaries and assigns private call-slot identities. There is no
public `Member` or `Source` wrapper.

### Option B · YAML

The equivalent [`fusion.yaml`](fusion.yaml) is:

```yaml
name: frontier-trio
models:
  - codex/gpt-5.5
  - gemini-cli/gemini-2.5-pro
  - anthropic/claude-sonnet-4-6
reducer:
  kind: majority_vote
  tie_breaker: codex/gpt-5.5
```

In [3]:
fusion_from_yaml = sf.Fusion.from_yaml("fusion.yaml")
fusion_from_yaml.url4 == fusion.url4

True

## 3 · Inspect the shareable recipe

The recipe contains model routes and an unresolved `$question`. Constructing or displaying it
sends nothing. Evaluation binds each concrete question later.

In [4]:
fusion.url4

"(panel_1=/codex/gpt-5.5()!'$question', panel_2=/gemini/2.5()!'$question', panel_3=/claude/sonnet-4.6()!'$question', {schema: 'screamingface.panel-result.v2', panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_id: 'anthropic/claude-sonnet-4-6', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'})"

## 4 · Run through the URL4 engine

For each question, ScreamingFace sends one complete expression to
`http://127.0.0.1:4404/v1`. The engine executes all three model routes and returns their labeled
answers. ScreamingFace never calls those routes, AI Gateway, or providers directly.

In [5]:
# For each question: GET http://127.0.0.1:4404/v1?q=<URL-encoded fusion expression>
# Decoded q expression:
# (question='<resolved GPQA prompt>',
#  panel_1=/codex/gpt-5.5()!'$question',
#  panel_2=/gemini/2.5()!'$question',
#  panel_3=/claude/sonnet-4.6()!'$question',
#  {schema: 'screamingface.panel-result.v2',
#   panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5',
#   panel_1_answer: '$panel_1',
#   panel_2_id: 'gemini-cli/gemini-2.5-pro',
#   panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2',
#   panel_3_id: 'anthropic/claude-sonnet-4-6',
#   panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'})
#
# Compiled URL4 request node (↖ shared = the same binding, not another request):
# GatherNode
# ├─ question: BindingNode → TextNode '<resolved GPQA prompt>'
# ├─ panel_1: BindingNode → RelUrlNode /codex/gpt-5.5
# │  ├─ context → empty
# │  └─ intent → question ↖ shared
# ├─ panel_2: BindingNode → RelUrlNode /gemini/2.5
# │  ├─ context → empty
# │  └─ intent → question ↖ shared
# ├─ panel_3: BindingNode → RelUrlNode /claude/sonnet-4.6
# │  ├─ context → empty
# │  └─ intent → question ↖ shared
# └─ response: StructNode
#    ├─ schema → screamingface.panel-result.v2
#    ├─ panel_1_id → codex/gpt-5.5
#    ├─ panel_1_model → codex/gpt-5.5
#    ├─ panel_1_answer → panel_1 ↖ shared
#    ├─ panel_2_id → gemini-cli/gemini-2.5-pro
#    ├─ panel_2_model → gemini-cli/gemini-2.5-pro
#    ├─ panel_2_answer → panel_2 ↖ shared
#    ├─ panel_3_id → anthropic/claude-sonnet-4-6
#    ├─ panel_3_model → anthropic/claude-sonnet-4-6
#    └─ panel_3_answer → panel_3 ↖ shared
run = fusion.evaluate("gpqa", first=20, seed=0)
run

Run(benchmark='GPQA-shaped synthetic science fixture', dataset_source='synthetic-gpqa-shaped', mode='mock', models=('codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6'), url="(panel_1=/codex/gpt-5.5()!'$question', panel_2=/gemini/2.5()!'$question', panel_3=/claude/sonnet-4.6()!'$question', {schema: 'screamingface.panel-result.v2', panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_id: 'anthropic/claude-sonnet-4-6', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'})", sample_size=20, seed=0, score=100.0, baseline=80.0, gain=20.0, cost_usd=0.0, fusion_name='frontier-trio', reducer='majority_vote', tie_breaker='codex/gpt-5.5', incomplete=0, profiles=(), pricing_source='engine response does not yet report usage', pricing_as_of='n/a', prompt_tokens=0, completion_tokens=0, total_tokens=0, model_results=(ModelResult(model='codex/gpt-5.5', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None), ModelResult(model='gemini-cli/gemini-2.5-pro', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None), ModelResult(model='anthropic/claude-sonnet-4-6', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None)), failures=())

## 5 · Compare

In [6]:
{
    "sample_size": run.sample_size,
    "score": run.score,
    "baseline": run.baseline,
    "gain": run.gain,
}

{'sample_size': 20, 'score': 100.0, 'baseline': 80.0, 'gain': 20.0}

> `gain = fusion score − best model score` on the same answers. Positive gain
means the combination corrected mistakes made by every individual panel model.

The URL4 engine owns model execution. ScreamingFace owns majority vote, answer-key scoring,
baseline, and gain. Real AI-Gateway-backed model routes can replace the deterministic commands
later without changing this SDK flow.

## 6 · Replace voting with one model-backed reducer

`Reducer` is the common contract; users construct concrete mechanisms. `MajorityVote` runs
deterministically in the SDK. `ModelReducer` adds one later URL4 model call whose prompt can
synthesize, select, rank, merge, or adjudicate the resolved panel answers.

The `$panel_answers` binding contains stable private slot IDs, model IDs, and resolved answers. The
reducer receives it in its URL4 intent, with empty context.

In [7]:
model_reduced = sf.Fusion(
    "frontier-trio-model-reduced",
    models=fusion.models,
    reducer=sf.ModelReducer(
        model="codex/gpt-5.5",
        prompt="Synthesize one final answer for $question from $panel_answers",
        params={"temperature": 0.0, "max_tokens": 512},
    ),
)
model_reduced.url4

"(panel_1=/codex/gpt-5.5()!'$question', panel_2=/gemini/2.5()!'$question', panel_3=/claude/sonnet-4.6()!'$question', panel_answers={panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_id: 'anthropic/claude-sonnet-4-6', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'}, fusion_answer=/codex/gpt-5.5?temperature=0.0&max_tokens=512&q=()!'Synthesize one final answer for $question from $panel_answers', {schema: 'screamingface.fusion-result.v2', panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_id: 'anthropic/claude-sonnet-4-6', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3', reducer: 'model', reducer_model: 'codex/gpt-5.5', answer

In [8]:
model_run = model_reduced.evaluate("gpqa", first=3, seed=0)
model_run

Run(benchmark='GPQA-shaped synthetic science fixture', dataset_source='synthetic-gpqa-shaped', mode='mock', models=('codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6'), url="(panel_1=/codex/gpt-5.5()!'$question', panel_2=/gemini/2.5()!'$question', panel_3=/claude/sonnet-4.6()!'$question', panel_answers={panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_id: 'anthropic/claude-sonnet-4-6', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'}, fusion_answer=/codex/gpt-5.5?temperature=0.0&max_tokens=512&q=()!'Synthesize one final answer for $question from $panel_answers', {schema: 'screamingface.fusion-result.v2', panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_id: 'anthropic/claude-sonnet-4-6', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3', reducer: 'model', reducer_model: 'codex/gpt-5.5', answer: '$fusion_answer'})", sample_size=3, seed=0, score=100.0, baseline=100.0, gain=0.0, cost_usd=0.0, fusion_name='frontier-trio-model-reduced', reducer='model', tie_breaker=None, incomplete=0, profiles=(), pricing_source='engine response does not yet report usage', pricing_as_of='n/a', prompt_tokens=0, completion_tokens=0, total_tokens=0, model_results=(ModelResult(model='codex/gpt-5.5', score=0.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None), ModelResult(model='gemini-cli/gemini-2.5-pro', score=100.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None), ModelResult(model='anthropic/claude-sonnet-4-6', score=100.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None)), failures=())